In [4]:
from hybrid_joint import *
from diffrax import *
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from controls import *

SyntaxError: invalid syntax (controls.py, line 38)

Same equation as in `dynamics_rough_vol.ipynb`, but with `M` trajectories.

In [ ]:
# global parameters

H = 0.1
M = 1000
grid_points = 10000
kappa = 1
T = 1
rho01, rho02, rho12 = 0.6, 0.4, 0.5
epsilon = 1/grid_points
times = jnp.arange(0, 1, epsilon)
lag = epsilon
solver_epsilon = 0.1*epsilon

In [ ]:
# Don't run! 30k sample paths take > 1 minute on my laptop

#hybrid_scheme(grid_points, 30000, T, H, kappa)

In [ ]:
# Don't run! 3 x 10k sample paths take 41 seconds

#for i in range(10000):
#    fbm, bm1, bm2 = correlated_fbm_bm_bm(0.8, 0, 0, grid_points, T, 0.1, kappa)

In [ ]:
# Let's parallelise the creation of the iid noise paths
# See the file hybrid_joint for the options regarding different correlation structures. Different functions are needed for fully correlated noise

from concurrent.futures import ProcessPoolExecutor

num_processes = 8  # Adjust based on your machine's CPU cores and task requirements

args = (rho01, rho02, rho12, grid_points, T, kappa)

with ProcessPoolExecutor(max_workers=num_processes) as executor:
    futures = [executor.submit(correlated_fbm_bm_bm_wrapper, args) for _ in range(M)]
    results = [future.result() for future in futures]

# 3 x 10k sample paths takes 10 seconds on my laptop, much better

# keep in mind though, unless the rho's are parameters this only needs to be run once for all passes of the calibration (unless there is something to be
# gained from using independent noises across different calibration steps)

# it's worth checking that the different sample paths are indeed independent

list_fbm, list_bm1, list_bm2 = zip(*results)

stacked_fbm = jnp.stack(list_fbm).T
stacked_bm1 = jnp.stack(list_bm1).T
stacked_bm2 = jnp.stack(list_bm2).T

In [ ]:
sample = 5

plt.figure(figsize=(20, 5))
plt.plot(times, stacked_fbm[:,sample], color = 'red')
plt.plot(times, stacked_bm1[:,sample], color = 'green')
plt.plot(times, stacked_bm2[:,sample], color = 'blue')

In [ ]:
# define the controls

bm1_interp = LinearInterpolation(times, stacked_bm1)
bm2_interp = LinearInterpolation(times, stacked_bm2)
fbm_interp = LinearInterpolation(times, stacked_fbm)

bm1_control = evaluate_and_reshape(bm1_interp.evaluate)
bm2_control = evaluate_and_reshape(bm2_interp.evaluate)
fbm_control = evaluate_and_reshape(fbm_interp.evaluate)
fbm_fun = lambda t: fbm_interp.evaluate(0,t)
fbm_lagged_control_temp = lagged_control(fbm_fun, lag)
fbm_lagged_control = lambda s,t: fbm_lagged_control_temp(s,t).reshape(1, M)
bm_control = stack_controls(bm1_control, bm2_control)
lead_lag_control = stack_controls(fbm_lagged_control, bm_control)

#check shape
lead_lag_control(0,0.5)[:,5].shape

# this is a function that returns a jnp array with the following structure: lead_lag_control(s,t)[lagged_fbm, bm1, bm2; sample_paths]

$$ \begin{cases} \mathrm{d}S_t = \sqrt{V_t} S_t \mathrm{d}\underline W^2_t \\ \mathrm{d}V_t = (\alpha + \beta V_t)\mathrm{d}t + \sigma_0 V_t^{\gamma_0} \mathrm{d} \underline B^H_t + \sigma_1 V_t^{\gamma_1} \mathrm{d}\underline W^1_t \end{cases} $$

In [ ]:
# define the SDE
# these parameters could be put inside the args of drift and diffusion
gamma0 = 1
gamma1 = 1
sigma0 = 0.1
sigma1 = 0.1
alpha = 1
beta = 1

saveat = SaveAt(ts = times)

# the drift, diffusion and initial condition of the RDE. This can be changed to any other RDE. In particular, if one wants the vol to be
# driven by a single noise, just set the corresponding term in the vector field to zero
y0 = jnp.array([1, 1])
drift = lambda t, y, args: jnp.array([0, alpha + beta*y[1]])
diffusion = lambda t, y, args: jnp.array([[0, 0, jnp.sqrt(y[1])*y[0]], [sigma0*jnp.power(y[1], gamma0), sigma1*jnp.power(y[1], gamma1), 0]])


In [ ]:
solutions = vmap_batch_fbm_solve(jnp.arange(M), lead_lag_control, drift, diffusion, y0, Midpoint(), 0, 1, solver_epsilon, saveat)

In [ ]:
# plot a single trajectory of the price and vol
plt.plot(solutions.ts[0,:], solutions.ys[0, :, 1], color = "red")
plt.plot(solutions.ts[0,:], solutions.ys[0, :, 0], color = "blue")

In [ ]:
# plot all trajectories of the price

import numpy as np
import matplotlib.pyplot as plt

# Assuming M and solutions are defined
for k in range(10):
    # Plot using the next color in the default cycle
    plt.plot(solutions.ts[0, :], solutions.ys[k, :, 0],linewidth=1)

plt.ylim(0, 10)
plt.show()